# Session 2: LangChain Development & Tool Integration (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 2. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks. Use this as your reference during class delivery.

**Consulting Context:** Throughout this session, all examples use McKinsey-style consulting scenarios -- market sizing, competitor analysis, client engagement workflows, and strategic frameworks like MECE and value chain analysis.

## Learning Objectives

By the end of this session, students will be able to:

1. **Use ChatModels and PromptTemplates** as LangChain's core building blocks
2. **Compose chains with LCEL** using the pipe operator (`|`)
3. **Create custom tools** using the `@tool` decorator
4. **Load and split documents** for retrieval workflows
5. **Build a RAG pipeline** that grounds LLM answers in external knowledge
6. **Manage conversation memory** across multi-turn interactions

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json
import os

print("All imports successful!")

All imports successful!


---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: LangChain Basics — ChatModels and PromptTemplates

ChatModels wrap LLM APIs into a unified interface. PromptTemplates let you parameterize prompts with variables so they become reusable across different inputs.

**Consulting scenario:** A McKinsey engagement team needs to quickly generate structured analyses across different industries and strategy frameworks. PromptTemplates let us build reusable "analysis templates" that any consultant can invoke with different client parameters.

In [12]:
# Demo 1 - LangChain Basics: ChatModels and PromptTemplates

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

response = llm.invoke("What is McKinsey's MECE framework in one sentence?")
print("Simple invocation:")
print(f"  Type: {type(response)}")
print(f"  Content: {response.content}")

messages = [
    SystemMessage(content="You are a McKinsey senior partner. Be concise and strategic."),
    HumanMessage(content="What is a value chain analysis?")
]
response = llm.invoke(messages)
print(f"\nWith messages:\n  {response.content}")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey consultant specializing in {domain}. Answer in {style} style."),
    ("human", "{question}")
])

formatted = prompt.format_messages(
    domain="growth strategy and market entry",
    style="executive briefing",
    question="What are the key drivers of growth in the EV battery market?"
)
print(f"\nFormatted prompt messages: {len(formatted)}")
for msg in formatted:
    print(f"  [{msg.type}]: {msg.content[:80]}...")

response = llm.invoke(formatted)
print(f"\nResponse: {response.content[:200]}...")

Simple invocation:
  Type: <class 'langchain_core.messages.ai.AIMessage'>
  Content: The MECE framework, developed by McKinsey & Company, stands for "Mutually Exclusive, Collectively Exhaustive," and is a problem-solving approach that ensures that information is organized in a way that avoids overlap and covers all possible options.

With messages:
  Value chain analysis is a strategic tool used to identify and evaluate the activities within an organization that create value for customers. It breaks down the company's operations into primary and support activities, allowing firms to understand how each contributes to competitive advantage and profitability. 

**Primary Activities** include:
1. Inbound Logistics
2. Operations
3. Outbound Logistics
4. Marketing and Sales
5. Service

**Support Activities** include:
1. Firm Infrastructure
2. Human Resource Management
3. Technology Development
4. Procurement

By analyzing these activities, companies can identify inefficiencies, optimize pro

### Demo 2: Building Chains with LCEL (Pipe Operator)

LCEL uses the `|` operator to compose components into chains. The pattern is: `prompt | model | parser`.

**Consulting scenario:** McKinsey engagement teams need multi-step analysis pipelines: research a market, analyze competitive dynamics, then produce an executive summary. LCEL lets us chain these steps cleanly -- mirroring how a real consulting workflow flows from data gathering through synthesis to recommendation.

In [13]:
# Demo 2 - Building Chains with LCEL

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey strategy consultant. Provide clear, structured insights."),
    ("human", "Explain {concept} in exactly {num_sentences} sentences, using a business consulting context.")
])
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"concept": "Porter's Five Forces", "num_sentences": "3"})
print("Simple chain result:")
print(result)

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output a JSON object with keys: framework_name, definition, consulting_use_case, complexity (1-10)."),
    ("human", "Describe the consulting framework: {concept}")
])
json_chain = json_prompt | llm | JsonOutputParser()
result = json_chain.invoke({"concept": "MECE principle"})
print(f"\nJSON chain result (type={type(result).__name__}):")
print(json.dumps(result, indent=2))

summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize this market research finding in one executive-ready sentence: {text}"
)
translate_prompt = ChatPromptTemplate.from_template(
    "Translate this executive summary to French for our Paris office: {text}"
)
summary_chain = summarize_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | llm | StrOutputParser()

text = "The global electric vehicle battery market is projected to grow at a CAGR of 19.2% from 2024 to 2030. Key growth drivers include government incentives, declining lithium-ion costs, and rising consumer demand for sustainable transportation. Asia-Pacific dominates with 68% market share, led by CATL and BYD."
summary = summary_chain.invoke({"text": text})
print(f"\nExecutive Summary: {summary}")
translation = translate_chain.invoke({"text": summary})
print(f"French (for Paris office): {translation}")

Simple chain result:
Porter's Five Forces framework is a strategic tool used to analyze the competitive environment of an industry by examining five key factors: the threat of new entrants, the bargaining power of suppliers, the bargaining power of buyers, the threat of substitute products or services, and the intensity of competitive rivalry. By assessing these forces, businesses can identify the underlying drivers of profitability and competitive dynamics, enabling them to develop strategies that enhance their market position. Ultimately, this analysis helps organizations make informed decisions regarding market entry, product development, and resource allocation.

JSON chain result (type=dict):
{
  "framework_name": "MECE Principle",
  "definition": "MECE stands for 'Mutually Exclusive, Collectively Exhaustive'. It is a problem-solving framework used in consulting to ensure that information is organized in a way that avoids overlap and covers all possible options. This principle hel

### Demo 3: Creating and Using Custom Tools

Tools let LLMs interact with the real world. Use the `@tool` decorator to turn any Python function into a tool.

**Consulting scenario:** McKinsey consultants rely on specialized analytical tools daily -- market sizing calculators, benchmark databases, financial ratio analyzers. Here we build custom tools that mirror these consulting utilities and bind them to an LLM so it can decide which tool to use based on a client question.

In [14]:
# Demo 3 - Creating and Using Custom Tools

@tool
def market_sizing_tool(population: float, adoption_rate: float, avg_revenue_per_user: float) -> dict:
    """Estimate total addressable market (TAM) given population, adoption rate (0-1), and average revenue per user."""
    tam = population * adoption_rate * avg_revenue_per_user
    return {
        "total_addressable_market": tam,
        "population": population,
        "adoption_rate_pct": f"{adoption_rate * 100:.1f}%",
        "arpu": avg_revenue_per_user,
        "formatted_tam": f"${tam:,.0f}"
    }

@tool
def benchmark_lookup_tool(metric: str) -> str:
    """Look up industry benchmark values for common consulting metrics."""
    benchmarks = {
        "saas_gross_margin": "70-85% gross margin is typical for SaaS companies",
        "retail_operating_margin": "3-5% operating margin is typical for retail",
        "pharma_r_and_d_spend": "15-25% of revenue is typical pharma R&D spend",
        "tech_revenue_growth": "20-40% YoY revenue growth for high-growth tech",
        "healthcare_ebitda_margin": "15-25% EBITDA margin for healthcare services",
    }
    return benchmarks.get(metric.lower(), f"No benchmark found for '{metric}'. Available: {list(benchmarks.keys())}")

@tool
def financial_ratio_tool(revenue: float, costs: float, assets: float) -> dict:
    """Calculate key financial ratios used in consulting engagements."""
    gross_margin = (revenue - costs) / revenue if revenue else 0
    asset_turnover = revenue / assets if assets else 0
    return {
        "gross_margin": f"{gross_margin:.1%}",
        "asset_turnover": f"{asset_turnover:.2f}x",
        "profit": f"${revenue - costs:,.0f}"
    }

print("Tool: market_sizing_tool")
print(f"  Name: {market_sizing_tool.name}")
print(f"  Description: {market_sizing_tool.description}")

print(f"\nmarket_sizing_tool (US EV market): {market_sizing_tool.invoke({'population': 330_000_000, 'adoption_rate': 0.08, 'avg_revenue_per_user': 45000})}")
print(f"benchmark_lookup_tool: {benchmark_lookup_tool.invoke('saas_gross_margin')}")
print(f"financial_ratio_tool: {financial_ratio_tool.invoke({'revenue': 50_000_000, 'costs': 32_000_000, 'assets': 80_000_000})}")

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
llm_with_tools = llm.bind_tools([market_sizing_tool, benchmark_lookup_tool, financial_ratio_tool])
response = llm_with_tools.invoke("What is the typical gross margin for a SaaS company?")
print(f"\nLLM response with tools:")
print(f"  Content: {response.content}")
print(f"  Tool calls: {response.tool_calls}")

Tool: market_sizing_tool
  Name: market_sizing_tool
  Description: Estimate total addressable market (TAM) given population, adoption rate (0-1), and average revenue per user.

market_sizing_tool (US EV market): {'total_addressable_market': 1188000000000.0, 'population': 330000000.0, 'adoption_rate_pct': '8.0%', 'arpu': 45000.0, 'formatted_tam': '$1,188,000,000,000'}
benchmark_lookup_tool: 70-85% gross margin is typical for SaaS companies
financial_ratio_tool: {'gross_margin': '36.0%', 'asset_turnover': '0.62x', 'profit': '$18,000,000'}

LLM response with tools:
  Content: 
  Tool calls: [{'name': 'benchmark_lookup_tool', 'args': {'metric': 'gross margin for SaaS companies'}, 'id': 'call_WkU7x9hQwXA1LAaf5ao4Kzso', 'type': 'tool_call'}]


### Demo 4: Document Loading and Text Splitting

RAG starts with loading documents and splitting them into manageable chunks.

**Consulting scenario:** McKinsey maintains vast knowledge bases -- industry reports, client case studies, best practice documents. Before we can retrieve relevant insights, we need to split these documents into chunks that fit within LLM context windows while preserving analytical coherence.

In [15]:
# Demo 4 - Document Loading and Text Splitting

documents = [
    Document(
        page_content="""McKinsey Global Institute research shows that generative AI could add $2.6 to $4.4 trillion
annually to the global economy across 63 use cases. The banking, high-tech, and life sciences
sectors stand to benefit most, with potential productivity gains of 3-5% of total industry revenue.
Organizations that move quickly on adoption will capture disproportionate value.""",
        metadata={"source": "mgi_genai_report.pdf", "chapter": "Executive Summary"}
    ),
    Document(
        page_content="""Post-merger integration (PMI) remains the most critical phase of any M&A transaction.
McKinsey research indicates that 70% of mergers fail to achieve expected synergies, primarily due to
cultural misalignment and slow integration execution. Best-practice PMI follows a 100-day plan
covering Day 1 readiness, synergy capture, operating model design, and cultural integration.""",
        metadata={"source": "mckinsey_ma_playbook.pdf", "chapter": "Post-Merger Integration"}
    )
]

print(f"Loaded {len(documents)} consulting documents")
for doc in documents:
    print(f"  Source: {doc.metadata['source']} | Chapter: {doc.metadata['chapter']} | Length: {len(doc.page_content)} chars")

splitter = RecursiveCharacterTextSplitter(chunk_size=int(os.getenv("CHUNK_SIZE", "200")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")), separators=["\n\n", "\n", ". ", " ", ""])
chunks = splitter.split_documents(documents)
print(f"\nSplit into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1} ({len(chunk.page_content)} chars) [{chunk.metadata['source']}]: {chunk.page_content[:80]}...")

Loaded 2 consulting documents
  Source: mgi_genai_report.pdf | Chapter: Executive Summary | Length: 366 chars
  Source: mckinsey_ma_playbook.pdf | Chapter: Post-Merger Integration | Length: 374 chars

Split into 4 chunks:
  Chunk 1 (285 chars) [mgi_genai_report.pdf]: McKinsey Global Institute research shows that generative AI could add $2.6 to $4...
  Chunk 2 (80 chars) [mgi_genai_report.pdf]: Organizations that move quickly on adoption will capture disproportionate value....
  Chunk 3 (281 chars) [mckinsey_ma_playbook.pdf]: Post-merger integration (PMI) remains the most critical phase of any M&A transac...
  Chunk 4 (92 chars) [mckinsey_ma_playbook.pdf]: covering Day 1 readiness, synergy capture, operating model design, and cultural ...


### Demo 5: Building a Simple RAG Pipeline

Knowledge base + retrieval + LCEL chain = a grounded Q&A system.

**Consulting scenario:** Imagine a McKinsey knowledge management system where consultants can query the firm's repository of industry insights, frameworks, and engagement best practices. The RAG pipeline retrieves the most relevant knowledge snippets and generates a grounded, evidence-based answer -- just like a seasoned partner drawing on decades of firm knowledge.

In [16]:
# Demo 5 - Building a Simple RAG Pipeline

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# McKinsey consulting knowledge base
knowledge_base = [
    "The MECE framework (Mutually Exclusive, Collectively Exhaustive) is McKinsey's foundational structuring principle. It ensures problem decomposition has no overlaps and no gaps, enabling rigorous hypothesis-driven analysis.",
    "McKinsey's Three Horizons of Growth framework helps companies allocate resources: Horizon 1 (core business), Horizon 2 (emerging opportunities), Horizon 3 (transformational initiatives).",
    "Value chain analysis, developed by Michael Porter, maps the primary and support activities that create value for customers. McKinsey consultants use it to identify cost reduction and differentiation opportunities.",
    "The McKinsey 7S Framework examines seven interdependent elements: Strategy, Structure, Systems, Shared Values, Style, Staff, and Skills. It is used for organizational effectiveness assessments.",
    "Digital transformation engagements typically follow McKinsey's 'rewire' approach: reimagine the business model, rebuild the technology foundation, and rewire the organization for speed and agility.",
    "Post-merger integration (PMI) success depends on capturing synergies within the first 100 days. McKinsey's PMI approach covers Day 1 readiness, clean rooms, synergy tracking, and cultural integration.",
]

def simple_retrieve(query, k=2):
    scored = []
    query_words = set(query.lower().split())
    for chunk in knowledge_base:
        chunk_words = set(chunk.lower().split())
        overlap = len(query_words & chunk_words)
        scored.append((overlap, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [chunk for _, chunk in scored[:k]]

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey knowledge management assistant. Answer based ONLY on the context from the firm's knowledge base. If not found, say 'This topic is not covered in the current knowledge base.'\n\nContext:\n{context}"),
    ("human", "{question}")
])
rag_chain = rag_prompt | llm | StrOutputParser()

for question in ["What is the MECE framework?", "How does McKinsey approach post-merger integration?", "What is the capital asset pricing model?"]:
    retrieved = simple_retrieve(question)
    context = "\n".join(f"- {c}" for c in retrieved)
    answer = rag_chain.invoke({"context": context, "question": question})
    print(f"Q: {question}\nA: {answer}\n")

Q: What is the MECE framework?
A: The MECE framework (Mutually Exclusive, Collectively Exhaustive) is McKinsey's foundational structuring principle. It ensures that problem decomposition has no overlaps and no gaps, enabling rigorous hypothesis-driven analysis.

Q: How does McKinsey approach post-merger integration?
A: McKinsey's approach to post-merger integration (PMI) focuses on capturing synergies within the first 100 days. Key components of this approach include:

1. **Day 1 Readiness**: Ensuring that the organization is prepared for the immediate changes that come with the merger.
2. **Clean Rooms**: Establishing secure environments for sensitive information to facilitate collaboration between merging entities.
3. **Synergy Tracking**: Monitoring and measuring the realization of synergies to ensure that the expected benefits of the merger are achieved.
4. **Cultural Integration**: Addressing the cultural differences between the merging organizations to foster a cohesive working e

---
## Tasks -- Full Solutions

Below are the complete, working solutions for all four student tasks.

### Task 1: Build a Consulting Analysis Chain with Prompt Templates and Output Parsers

**Approach:** We create a ChatPromptTemplate that instructs the model to output JSON with a structured consulting analysis. The chain takes an industry and analysis type, generates a MECE-structured insight with key findings and a strategic recommendation, and returns parsed JSON. The system message defines the output schema.

**Consulting scenario:** An engagement manager needs to quickly generate structured industry analyses for different sectors and strategy questions. The chain produces a standardized JSON output that can be fed into client-ready slide decks.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 8-10 minutes. This is students' first LCEL chain. Write the pipe syntax on the board: `prompt | llm | parser`. Emphasize that LCEL is just Python's `|` operator overloaded for composition. Suggest starting with `StrOutputParser` first, then switching to `JsonOutputParser`. After the exercise, discuss why JSON output is important for downstream processing in agentic systems.
>
> **Common Student Mistakes:**
> - Forgetting to import `JsonOutputParser` from `langchain_core.output_parsers`
> - Not including 'JSON' or format instructions in the prompt -- the model returns prose instead of JSON
> - Calling `chain.run()` (old API) instead of `chain.invoke()` (LCEL API)
> - Passing a string instead of a dictionary to `chain.invoke()` when the prompt has multiple variables
> - Not understanding that `|` creates a new chain object -- it doesn't modify in place
>
> **Skippable?** NO -- CRITICAL -- LCEL is the foundation for all LangChain chains. Every subsequent task and Day 3 RAG pipeline uses this pattern. Do NOT skip.

In [17]:
# Task 1 - SOLUTION: Build a Consulting Analysis Chain with Prompt Templates and Output Parsers

def create_consulting_analysis_chain():
    """Create a chain that generates structured consulting analyses."""
    llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are a McKinsey senior consultant. Output a JSON object with these fields: "
            "industry (string), analysis_type (string), "
            "key_findings (list of 3 strings, each a MECE-structured insight), "
            "strategic_recommendation (string with a clear action item), "
            "risk_factors (list of 2 strings). "
            "Ensure findings are mutually exclusive and collectively exhaustive."
        )),
        ("human", "Generate a {analysis_type} analysis for the {industry} industry.")
    ])

    chain = prompt | llm | JsonOutputParser()
    return chain


# Test
chain = create_consulting_analysis_chain()
result = chain.invoke({"industry": "electric vehicle batteries", "analysis_type": "market entry"})
print(json.dumps(result, indent=2))

{
  "industry": "Electric Vehicle Batteries",
  "analysis_type": "Market Entry Analysis",
  "key_findings": [
    "The global demand for electric vehicle batteries is projected to grow at a CAGR of 20% over the next five years, driven by increasing EV adoption and government incentives.",
    "Current market leaders hold over 60% of the market share, indicating high barriers to entry due to established supply chains and brand loyalty.",
    "Technological advancements in battery chemistry, such as solid-state batteries, present opportunities for differentiation and potential market disruption."
  ],
  "strategic_recommendation": "Establish partnerships with automotive manufacturers and invest in R&D for next-generation battery technologies to enhance competitive positioning.",
  "risk_factors": [
    "Volatility in raw material prices, particularly lithium and cobalt, which can impact production costs and margins.",
    "Regulatory changes and environmental policies that may affect man

### Task 2: Create Custom Consulting Tools

**Approach:** Each tool is a decorated Python function with a clear docstring that the LLM uses to decide when to invoke it. The `competitor_analysis_tool` performs a structured competitive comparison, `market_sizing_tool` estimates TAM/SAM/SOM, and `engagement_roi_tool` calculates projected ROI for a consulting engagement. We then bind all three to an LLM.

**Consulting scenario:** These tools represent the analytical utilities that a McKinsey team would use during a typical engagement -- sizing markets, analyzing competitors, and projecting engagement value for the client.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 8-10 minutes. Walk through the `@tool` decorator syntax from Demo 3. Suggest students implement one tool first, test it, then add the other two. After the exercise, show `llm_with_tools.invoke()` and discuss how the model decides which tool to call. Ask: 'What happens if a query doesn't match any tool?' (Answer: the model responds directly without calling tools).
>
> **Common Student Mistakes:**
> - Not adding docstrings to `@tool` decorated functions -- LangChain uses the docstring as the tool description for the LLM
> - Incorrect type annotations on tool parameters -- LangChain infers the schema from type hints
> - Returning complex objects instead of strings from tools -- tool outputs must be serializable
> - Trying to call `tool.invoke()` directly instead of letting the LLM decide when to use the tool
>
> **Skippable?** YES -- CAN SKIP if running behind schedule. The `@tool` pattern is well-covered in Demo 3. If skipping, briefly show one custom tool and how `bind_tools()` works, then move to Task 3 (memory) which is more foundational for agents.

In [18]:
# Task 2 - SOLUTION: Create Custom Consulting Tools

@tool
def competitor_analysis_tool(company: str, competitors: list[str]) -> dict:
    """Perform a structured competitor analysis comparing a company against its competitors across key dimensions."""
    analysis = {
        "focal_company": company,
        "competitors_analyzed": len(competitors),
        "competitor_list": competitors,
        "dimensions": ["Market Share", "Product Portfolio", "Geographic Reach", "Innovation Pipeline"],
        "methodology": "McKinsey Competitive Positioning Framework",
        "recommendation": f"Conduct deep-dive on {competitors[0] if competitors else 'top competitor'} as primary competitive threat to {company}"
    }
    return analysis

@tool
def market_sizing_estimate(total_population: float, serviceable_pct: float, obtainable_pct: float, avg_deal_value: float) -> dict:
    """Estimate Total Addressable Market (TAM), Serviceable Addressable Market (SAM), and Serviceable Obtainable Market (SOM)."""
    tam = total_population * avg_deal_value
    sam = tam * serviceable_pct
    som = sam * obtainable_pct
    return {
        "TAM": f"${tam:,.0f}",
        "SAM": f"${sam:,.0f}",
        "SOM": f"${som:,.0f}",
        "methodology": "Top-down market sizing with TAM/SAM/SOM framework"
    }

@tool
def engagement_roi_tool(engagement_cost: float, projected_savings: float, implementation_months: int) -> dict:
    """Calculate the projected ROI and payback period for a McKinsey consulting engagement."""
    roi = ((projected_savings - engagement_cost) / engagement_cost) * 100 if engagement_cost else 0
    monthly_savings = projected_savings / 12
    payback_months = engagement_cost / monthly_savings if monthly_savings else 0
    return {
        "engagement_cost": f"${engagement_cost:,.0f}",
        "projected_annual_savings": f"${projected_savings:,.0f}",
        "roi_pct": f"{roi:.1f}%",
        "payback_period_months": f"{payback_months:.1f}",
        "implementation_timeline": f"{implementation_months} months",
        "verdict": "Strong ROI" if roi > 200 else "Moderate ROI" if roi > 100 else "Marginal ROI"
    }


# Test tools directly
print(f"competitor_analysis_tool: {competitor_analysis_tool.invoke({'company': 'Tesla', 'competitors': ['BYD', 'Rivian', 'Lucid']})}")
print(f"\nmarket_sizing_estimate: {market_sizing_estimate.invoke({'total_population': 5_000_000, 'serviceable_pct': 0.3, 'obtainable_pct': 0.1, 'avg_deal_value': 50_000})}")
print(f"\nengagement_roi_tool: {engagement_roi_tool.invoke({'engagement_cost': 2_000_000, 'projected_savings': 15_000_000, 'implementation_months': 6})}")

# Bind to LLM
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
llm_with_tools = llm.bind_tools([competitor_analysis_tool, market_sizing_estimate, engagement_roi_tool])
response = llm_with_tools.invoke("Estimate the market size for premium electric SUVs with 10 million potential buyers, 25% serviceable, 8% obtainable, and $65,000 average price")
print(f"\nLLM tool calls: {response.tool_calls}")

competitor_analysis_tool: {'focal_company': 'Tesla', 'competitors_analyzed': 3, 'competitor_list': ['BYD', 'Rivian', 'Lucid'], 'dimensions': ['Market Share', 'Product Portfolio', 'Geographic Reach', 'Innovation Pipeline'], 'methodology': 'McKinsey Competitive Positioning Framework', 'recommendation': 'Conduct deep-dive on BYD as primary competitive threat to Tesla'}

market_sizing_estimate: {'TAM': '$250,000,000,000', 'SAM': '$75,000,000,000', 'SOM': '$7,500,000,000', 'methodology': 'Top-down market sizing with TAM/SAM/SOM framework'}

engagement_roi_tool: {'engagement_cost': '$2,000,000', 'projected_annual_savings': '$15,000,000', 'roi_pct': '650.0%', 'payback_period_months': '1.6', 'implementation_timeline': '6 months', 'verdict': 'Strong ROI'}

LLM tool calls: [{'name': 'market_sizing_estimate', 'args': {'total_population': 10000000, 'serviceable_pct': 0.25, 'obtainable_pct': 0.08, 'avg_deal_value': 65000}, 'id': 'call_Z7CSFiUBXrn6njuK0eBA5z0Y', 'type': 'tool_call'}]


### Task 3: Implement a Conversational Consulting Advisor with Memory

**Approach:** The `ConsultingAdvisor` class uses a `MessagesPlaceholder` in the prompt to inject chat history. Each call appends HumanMessage and AIMessage objects to the history list, giving the LLM full context of the engagement discussion. The system prompt positions the LLM as a McKinsey partner guiding a client through strategic options.

**Consulting scenario:** A client CEO is exploring strategic options for their company. The conversational advisor remembers previous discussion points -- just like a real McKinsey partner would across multiple steering committee meetings.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 10-12 minutes. This is conceptually important -- it bridges Session 1's SimpleAgent to LangChain's memory system. Emphasize that `MessagesPlaceholder('history')` is where prior turns get injected. Suggest students get a single-turn response working first, then add the history list and multi-turn. After the exercise, test with 3+ turns to show the model remembering context.
>
> **Common Student Mistakes:**
> - Forgetting `MessagesPlaceholder('history')` in the prompt template -- the history is never injected
> - Passing `self.history` as a string instead of a list of `HumanMessage`/`AIMessage` objects
> - Appending `HumanMessage` but forgetting to append `AIMessage` -- the model loses assistant context
> - Using `ChatPromptTemplate.from_template()` (single template) instead of `ChatPromptTemplate.from_messages()` (message list)
> - Not passing `{'history': self.history, ...}` to `chain.invoke()` -- the history variable is missing
>
> **Skippable?** NO -- CRITICAL -- conversation memory is essential for multi-turn agents. Day 2 Sessions 3-4 and Day 3 all assume students understand this pattern. Do NOT skip.

In [19]:
# Task 3 - SOLUTION: Implement a Conversational Consulting Advisor with Memory

class ConsultingAdvisor:
    def __init__(self):
        """Initialize the consulting advisor chain with memory."""
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", (
                "You are a McKinsey senior partner advising a client CEO on strategic decisions. "
                "Draw on McKinsey frameworks (MECE, 7S, Three Horizons) where appropriate. "
                "Be structured, insightful, and action-oriented. Remember the full engagement history."
            )),
            MessagesPlaceholder("history"),
            ("human", "{input}")
        ])
        self.chain = self.prompt | self.llm | StrOutputParser()
        self.history = []

    def chat(self, user_input):
        """Send a message and get a strategic response, maintaining engagement history."""
        response = self.chain.invoke({
            "history": self.history,
            "input": user_input
        })
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=response))
        return response

    def get_history(self):
        """Return the engagement conversation history."""
        return self.history


# Test -- simulating a client engagement conversation
advisor = ConsultingAdvisor()
print("CLIENT CEO: We are a mid-size healthcare company considering acquiring a smaller biotech firm to expand our oncology pipeline.")
print(f"MCKINSEY PARTNER: {advisor.chat('We are a mid-size healthcare company considering acquiring a smaller biotech firm to expand our oncology pipeline. What should we consider?')}")
print()
print("CLIENT CEO: What are the biggest post-merger integration risks we should watch for?")
print(f"MCKINSEY PARTNER: {advisor.chat('What are the biggest post-merger integration risks we should watch for given our situation?')}")
print(f"\nEngagement history: {len(advisor.get_history())} messages")

CLIENT CEO: We are a mid-size healthcare company considering acquiring a smaller biotech firm to expand our oncology pipeline.
MCKINSEY PARTNER: Acquiring a smaller biotech firm to enhance your oncology pipeline is a strategic move that can provide significant growth opportunities. To ensure a successful acquisition, we can apply several McKinsey frameworks to structure our analysis and decision-making process. Here’s a structured approach:

### 1. **Strategic Rationale (Three Horizons Framework)**

**Horizon 1: Core Business**
- **Current Portfolio Assessment**: Evaluate your existing oncology pipeline. What are the strengths and weaknesses? How does the potential acquisition align with your current offerings?
- **Market Position**: Analyze your current market position in oncology. How will the acquisition enhance your competitive advantage?

**Horizon 2: Emerging Opportunities**
- **Pipeline Synergies**: Identify how the biotech firm’s products or technologies can complement or enhan

### Task 4: Build a RAG-Powered Consulting Knowledge Base

**Approach:** The `ConsultingRAG` class accepts Document objects representing McKinsey industry reports and frameworks, splits them with `RecursiveCharacterTextSplitter`, and stores the chunks. The `retrieve` method uses keyword overlap scoring. The `ask` method retrieves context, includes source metadata, and uses an LCEL chain to generate a grounded, partner-quality answer.

**Consulting scenario:** Build a knowledge management system that consultants can query to find relevant insights from McKinsey's proprietary industry research and strategy frameworks.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 10-12 minutes. This is a preview of Day 3's full RAG session. Walk students through the retrieve-then-generate pattern. The keyword-overlap scoring is intentionally simple -- tell students: 'On Day 3 you'll replace this with embedding-based semantic search, but the retrieve -> generate architecture stays the same.' Focus on the RAG chain composition with LCEL.
>
> **Common Student Mistakes:**
> - Implementing retrieval as a simple string match instead of keyword overlap scoring
> - Not including source metadata in the context passed to the LLM -- the model can't cite sources
> - Returning raw chunks without any formatting -- the LLM needs clearly separated context pieces
> - Setting `chunk_overlap=0` which causes information loss at chunk boundaries
> - Not testing with queries that span multiple documents to verify retrieval quality
>
> **Skippable?** YES -- CAN SKIP if running behind schedule. This is a simplified preview of Day 3 Session 1 (full RAG). If skipping, explain the retrieve-then-generate concept verbally with a diagram and tell students they will build a proper version tomorrow with embeddings and ChromaDB.

In [20]:
# Task 4 - SOLUTION: Build a RAG-Powered Consulting Knowledge Base

class ConsultingRAG:
    def __init__(self, documents):
        """Initialize with McKinsey knowledge documents, split into chunks."""
        splitter = RecursiveCharacterTextSplitter(chunk_size=int(os.getenv("CHUNK_SIZE", "300")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "50")))
        self.chunks = splitter.split_documents(documents)
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", (
                "You are a McKinsey knowledge management assistant. "
                "Answer the question based ONLY on the provided context from firm research and reports. "
                "Include which source(s) you used. Structure your answer using MECE principles. "
                "If the context does not contain the answer, say so.\n\n"
                "Context:\n{context}"
            )),
            ("human", "{question}")
        ])
        self.chain = self.prompt | self.llm | StrOutputParser()
        print(f"Initialized ConsultingRAG with {len(self.chunks)} chunks from {len(documents)} documents")

    def retrieve(self, query, k=3):
        """Retrieve the top-k most relevant chunks for a consulting query."""
        scored = []
        query_words = set(query.lower().split())
        for chunk in self.chunks:
            chunk_words = set(chunk.page_content.lower().split())
            score = len(query_words & chunk_words)
            scored.append((score, chunk))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [chunk for _, chunk in scored[:k]]

    def ask(self, question):
        """Answer a consulting question using retrieved context from the knowledge base."""
        retrieved = self.retrieve(question)
        context_parts = []
        for chunk in retrieved:
            source = chunk.metadata.get('source', 'unknown')
            context_parts.append(f"[Source: {source}] {chunk.page_content}")
        context = "\n\n".join(context_parts)
        return self.chain.invoke({"context": context, "question": question})


# Test with McKinsey-style consulting documents
docs = [
    Document(
        page_content=(
            "The global EV battery market is projected to reach $360 billion by 2030, growing at a CAGR of 19.2%. "
            "Key growth drivers include government incentives (IRA in the US, Green Deal in EU), declining lithium-ion costs, "
            "and rising consumer demand. Asia-Pacific dominates with 68% market share, led by CATL and BYD. "
            "Supply chain concentration in China remains the primary strategic risk for Western OEMs."
        ),
        metadata={"source": "mckinsey_ev_battery_outlook_2024.pdf"}
    ),
    Document(
        page_content=(
            "McKinsey's analysis of 2,500 M&A transactions reveals that successful acquirers share three traits: "
            "they conduct rigorous due diligence on cultural compatibility, they establish a dedicated integration "
            "management office (IMO) within 30 days, and they set aggressive but achievable 100-day synergy targets. "
            "Healthcare and technology sectors show the highest variance in post-merger outcomes, "
            "with cultural integration being the single largest predictor of success."
        ),
        metadata={"source": "mckinsey_ma_excellence_report.pdf"}
    ),
    Document(
        page_content=(
            "Digital transformation across financial services is accelerating. McKinsey estimates that banks "
            "investing in AI-driven operations can reduce costs by 20-30% while improving customer satisfaction scores "
            "by 15-25 points. The most successful transformations follow a 'rewire' approach: reimagine customer journeys, "
            "rebuild core technology platforms, and reorganize around agile, cross-functional teams."
        ),
        metadata={"source": "mckinsey_digital_banking_2024.pdf"}
    ),
]

rag = ConsultingRAG(docs)
print(rag.ask("What are the growth drivers in the EV battery market?"))
print()
print(rag.ask("What makes M&A integrations successful in healthcare?"))
print()
print(rag.ask("How can banks benefit from digital transformation?"))

Initialized ConsultingRAG with 6 chunks from 3 documents
The growth drivers in the EV battery market are as follows:

1. **Government Incentives**: Initiatives such as the Inflation Reduction Act (IRA) in the US and the Green Deal in the EU are promoting the adoption of electric vehicles and, consequently, the demand for EV batteries.

2. **Declining Lithium-Ion Costs**: The reduction in costs associated with lithium-ion batteries is making electric vehicles more affordable, thus driving market growth.

3. **Rising Consumer Demand**: There is an increasing consumer preference for electric vehicles, which is contributing to the growth of the EV battery market.

(Source: mckinsey_ev_battery_outlook_2024.pdf)

The provided context does not contain specific information regarding what makes M&A integrations successful in healthcare. The insights available focus on general M&A transactions and do not specify the healthcare sector. 

Sources used: mckinsey_ma_excellence_report.pdf.

Banks can

---
## Summary

In this session students learned the core LangChain building blocks, applied throughout to McKinsey consulting scenarios:

1. **ChatModels & PromptTemplates** -- How LangChain wraps LLM APIs and provides reusable, parameterized prompt templates for consulting analyses (market research, strategy frameworks, executive briefings).
2. **LCEL Chains** -- How the pipe operator (`|`) composes prompt, model, and parser into clean analysis pipelines (research to synthesis to executive summary).
3. **Custom Tools** -- How the `@tool` decorator turns Python functions into consulting tools (market sizing, competitor analysis, financial modeling, benchmark lookup) that LLMs can discover and invoke.
4. **Document Loading & Splitting** -- How to prepare McKinsey industry reports and knowledge base documents for retrieval by splitting them into overlapping chunks.
5. **RAG Pipelines** -- How to combine retrieval with generation to ground consulting answers in firm research and industry data.

**Instructor Notes:**
- Emphasize LCEL as the modern LangChain way -- discourage legacy `LLMChain` usage.
- For Task 2, show how the docstring becomes the tool description the LLM sees -- this maps to how consultants describe analytical methodologies.
- For Task 3, discuss what happens when engagement history grows beyond the context window -- parallels to real engagement knowledge management.
- For Task 4, note that keyword matching is a simplification -- production McKinsey systems would use vector similarity (embeddings) for semantic search.

**Up Next:** In Session 3, we will move from linear chains to graph-based workflows with LangGraph.